# Protobuf Basics — 01: What is Protocol Buffers?

Protocol Buffers (Protobuf) is Google's binary serialization format.
It is NOT a file format like Parquet or ORC — it is a message format
primarily used for RPC, gRPC services, and Kafka pipelines.

Topics: Protobuf vs JSON vs Avro, .proto schema definition,
field numbers, wire types, how Spark reads Protobuf data.


In [1]:
import os, time, pathlib, shutil, random, datetime, subprocess, glob as glib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/protobuf_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('protobuf-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/10 21:28:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2


## Step 1 — What is Protobuf and Why it Exists



In [2]:

print("""
Protocol Buffers (Protobuf):
  Created by Google for internal RPC (Remote Procedure Call)
  Binary format — much smaller and faster than JSON/XML
  Schema-first — schema (`.proto` file) defines message structure
  Language-neutral — generate code for Java, Python, Go, C++, etc.

.proto schema example:
  syntax = "proto3";
  message Order {
    int64  order_id    = 1;
    int64  customer_id = 2;
    string product     = 3;
    string category    = 4;
    double price       = 5;
    int32  quantity    = 6;
    string status      = 7;
  }

Key concepts:
  - Field numbers (= 1, = 2, ...): wire format uses these, NOT field names
  - Wire types: 0=varint, 1=64-bit, 2=length-delimited, 5=32-bit
  - Binary format: NOT human-readable (unlike JSON)
  - Schema required to deserialize (unlike JSON which is self-describing)
""")



Protocol Buffers (Protobuf):
  Created by Google for internal RPC (Remote Procedure Call)
  Binary format — much smaller and faster than JSON/XML
  Schema-first — schema (`.proto` file) defines message structure
  Language-neutral — generate code for Java, Python, Go, C++, etc.

.proto schema example:
  syntax = "proto3";
  message Order {
    int64  order_id    = 1;
    int64  customer_id = 2;
    string product     = 3;
    string category    = 4;
    double price       = 5;
    int32  quantity    = 6;
    string status      = 7;
  }

Key concepts:
  - Field numbers (= 1, = 2, ...): wire format uses these, NOT field names
  - Wire types: 0=varint, 1=64-bit, 2=length-delimited, 5=32-bit
  - Binary format: NOT human-readable (unlike JSON)
  - Schema required to deserialize (unlike JSON which is self-describing)



## Step 2 — Protobuf vs JSON vs Avro



In [3]:

print("""
Feature comparison:
  Feature              Protobuf      JSON         Avro
  ─────────────────────────────────────────────────────
  Format               Binary        Text         Binary
  Schema               .proto file   None (self)  JSON schema
  Schema in data       No            Yes          Yes (header)
  Human-readable       No            Yes          No
  Size                 Very small    Large        Medium
  Parse speed          Very fast     Slow         Fast
  Schema evolution     Good          N/A          Excellent
  Field names in data  No (numbers)  Yes          No (IDs)
  gRPC support         ✅ Native     ❌           ❌
  Kafka support        ✅ Common     ✅ Common    ✅ Standard
  Spark native         ❌ Needs JAR  ✅ Built-in  ✅ With JAR

Size comparison (same 1000-field order record):
  JSON     : ~200 bytes (field names + values)
  Avro     : ~80 bytes (no field names, binary)
  Protobuf : ~50 bytes (no field names, varint encoding, optional fields omitted)

When to use Protobuf:
  ✅ gRPC services (native transport format)
  ✅ Kafka + Confluent Schema Registry (Protobuf support)
  ✅ High-throughput microservice communication
  ✅ Mobile/IoT (bandwidth constrained)
  ✅ When you need smallest possible message size

When NOT to use:
  ❌ Analytics storage (use Parquet/ORC instead)
  ❌ Human-readable logs (use JSON)
  ❌ Ad-hoc data exchange (use JSON or CSV)
""")



Feature comparison:
  Feature              Protobuf      JSON         Avro
  ─────────────────────────────────────────────────────
  Format               Binary        Text         Binary
  Schema               .proto file   None (self)  JSON schema
  Schema in data       No            Yes          Yes (header)
  Human-readable       No            Yes          No
  Size                 Very small    Large        Medium
  Parse speed          Very fast     Slow         Fast
  Schema evolution     Good          N/A          Excellent
  Field names in data  No (numbers)  Yes          No (IDs)
  gRPC support         ✅ Native     ❌           ❌
  Kafka support        ✅ Common     ✅ Common    ✅ Standard
  Spark native         ❌ Needs JAR  ✅ Built-in  ✅ With JAR

Size comparison (same 1000-field order record):
  JSON     : ~200 bytes (field names + values)
  Avro     : ~80 bytes (no field names, binary)
  Protobuf : ~50 bytes (no field names, varint encoding, optional fields omitted)

When to

## Step 3 — How Spark Reads Protobuf



In [4]:

print("""
Spark 3.4+ has built-in Protobuf support via spark-protobuf.

Requirements:
  1. spark-protobuf JAR (included in Spark 3.4+)
     or: add to classpath: spark-protobuf_2.13-x.x.x.jar

  2. Two ways to specify schema:
     a) .proto descriptor file (compiled .desc binary)
     b) DDL schema string (when you know the structure)

  3. Spark reads Protobuf as binary column → deserializes to struct

Reading pattern:
  # From Kafka
  df.select(
      F.from_protobuf(F.col("value"), "MessageName", "/path/schema.desc").alias("data")
  ).select("data.*")

  # From file (binary Protobuf files)
  spark.read.format("protobuf")
       .option("protobufDescriptorFilePath", "/path/schema.desc")
       .option("messageName", "Order")
       .load("/path/to/protos/")

In our cluster (Spark 4.0.2):
  spark-protobuf is available
  Use: from pyspark.sql.protobuf.functions import from_protobuf, to_protobuf
""")



Spark 3.4+ has built-in Protobuf support via spark-protobuf.

Requirements:
  1. spark-protobuf JAR (included in Spark 3.4+)
     or: add to classpath: spark-protobuf_2.13-x.x.x.jar

  2. Two ways to specify schema:
     a) .proto descriptor file (compiled .desc binary)
     b) DDL schema string (when you know the structure)

  3. Spark reads Protobuf as binary column → deserializes to struct

Reading pattern:
  # From Kafka
  df.select(
      F.from_protobuf(F.col("value"), "MessageName", "/path/schema.desc").alias("data")
  ).select("data.*")

  # From file (binary Protobuf files)
  spark.read.format("protobuf")
       .option("protobufDescriptorFilePath", "/path/schema.desc")
       .option("messageName", "Order")
       .load("/path/to/protos/")

In our cluster (Spark 4.0.2):
  spark-protobuf is available
  Use: from pyspark.sql.protobuf.functions import from_protobuf, to_protobuf



## Step 4 — Protobuf in the Data Pipeline



In [5]:

print("""
Typical Protobuf pipeline in production:
  
  [gRPC Service / Producer]
        │ Serialize to Protobuf bytes
        ▼
  [Kafka Topic]
    key: bytes
    value: bytes (Protobuf)
        │
        ▼
  [Spark Structured Streaming]
    from_protobuf(col("value"), "Order", descriptor_path)
        │ Deserialize to DataFrame
        ▼
  [Parquet / Delta / Iceberg]
    Analytics zone (columnar, compressed)
        │
        ▼
  [BI / Analytics Queries]

The schema (.proto compiled to .desc) is the contract between producer and consumer.
Confluent Schema Registry supports Protobuf schemas alongside Avro.
""")



Typical Protobuf pipeline in production:
  
  [gRPC Service / Producer]
        │ Serialize to Protobuf bytes
        ▼
  [Kafka Topic]
    key: bytes
    value: bytes (Protobuf)
        │
        ▼
  [Spark Structured Streaming]
    from_protobuf(col("value"), "Order", descriptor_path)
        │ Deserialize to DataFrame
        ▼
  [Parquet / Delta / Iceberg]
    Analytics zone (columnar, compressed)
        │
        ▼
  [BI / Analytics Queries]

The schema (.proto compiled to .desc) is the contract between producer and consumer.
Confluent Schema Registry supports Protobuf schemas alongside Avro.



## Step 5 — Installing protobuf Python Library



In [6]:

import subprocess
result = subprocess.run(["pip3","install","--quiet","--break-system-packages","protobuf>=4.0"],
                       capture_output=True, text=True)
print(f"protobuf install: {'OK' if result.returncode==0 else result.stderr[:100]}")

try:
    import google.protobuf
    print(f"protobuf version: {google.protobuf.__version__}")
    print("✅ protobuf Python library available")
except ImportError:
    print("❌ protobuf not available — install with: pip install protobuf")
    print("   Subsequent notebooks will use simulated serialization where needed")


protobuf install: 
Usage:   
  pip3 install [options] <requirement specifier> [package-index-options] ...
  pip3 insta
❌ protobuf not available — install with: pip install protobuf
   Subsequent notebooks will use simulated serialization where needed


## Summary



In [7]:

print("""
Protobuf fundamentals:
  - Binary format — NOT human-readable
  - Schema defined in .proto files — REQUIRED for deserialization
  - Field numbers (not names) identify fields in binary format
  - Smaller and faster than JSON, comparable to Avro
  - Native format for gRPC; common in Kafka pipelines

In Spark 4.0.2:
  from pyspark.sql.protobuf.functions import from_protobuf, to_protobuf

  # Deserialize binary column
  df.withColumn("data", from_protobuf(col("value"), "MsgName", "path/to.desc"))

  # Serialize struct column
  df.withColumn("bytes", to_protobuf(col("struct_col"), "MsgName", "path/to.desc"))
""")



Protobuf fundamentals:
  - Binary format — NOT human-readable
  - Schema defined in .proto files — REQUIRED for deserialization
  - Field numbers (not names) identify fields in binary format
  - Smaller and faster than JSON, comparable to Avro
  - Native format for gRPC; common in Kafka pipelines

In Spark 4.0.2:
  from pyspark.sql.protobuf.functions import from_protobuf, to_protobuf

  # Deserialize binary column
  df.withColumn("data", from_protobuf(col("value"), "MsgName", "path/to.desc"))

  # Serialize struct column
  df.withColumn("bytes", to_protobuf(col("struct_col"), "MsgName", "path/to.desc"))

